# Âge cérébral sur IXI : effet du changement d'hôpital (ClinicaDL 2.0)

**Avant de commencer :** Exécution → Modifier le type d'exécution → **GPU T4**.

Organisation :
- les scripts viennent du dépôt GitHub [Ayoub-TAREK/ixi-brain-age](https://github.com/Ayoub-TAREK/ixi-brain-age) (`git clone` la première fois, `git pull` ensuite) ;
- les données sont téléchargées dans `/content` (rapide, mais effacé à la fin de la session) ;
- les résultats sont enregistrés sur Drive (`MyDrive/test-clinicadl/outputs/`), pour ne rien perdre si la session coupe.

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi
!python --version

## 2. Connecter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Récupérer les scripts depuis GitHub

À relancer après chaque `git push` depuis le PC pour avoir la dernière version.

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/test-clinicadl'
WORK = '/content/test-clinicadl'
REPO = 'https://github.com/Ayoub-TAREK/ixi-brain-age.git'

# un ancien dossier sans .git (ancienne version du notebook) empêcherait le clone
if os.path.exists(WORK) and not os.path.exists(f'{WORK}/.git'):
    !rm -rf {WORK}

if not os.path.exists(WORK):
    !git clone {REPO} {WORK}
else:
    !git -C {WORK} pull

os.makedirs(f'{WORK}/data/IXI/raw', exist_ok=True)
os.makedirs(f'{DRIVE}/outputs', exist_ok=True)
if not os.path.exists(f'{WORK}/outputs'):
    os.symlink(f'{DRIVE}/outputs', f'{WORK}/outputs')
!git -C {WORK} log --oneline -3
!ls {WORK}/scripts

## 3 bis. Unpack the t1-linear images (Clinica, 2 mm)

Upload `caps_t1linear_2mm.tar` (1.5 GB, made on the PC by `scripts/t1_linear_wsl.sh`) to `MyDrive/test-clinicadl/` once. This cell unpacks it into `data/IXI/caps_t1linear_2mm` (about 30 s) and is skipped if it is already there.

In [ ]:
import glob, os
T1L_TAR = f'{DRIVE}/caps_t1linear_2mm.tar'
T1L_DIR = f'{WORK}/data/IXI/caps_t1linear_2mm'
if not glob.glob(f'{T1L_DIR}/subjects/*/*/t1_linear/*_T1w.nii.gz'):
    assert os.path.exists(T1L_TAR), f'{T1L_TAR} not found: upload caps_t1linear_2mm.tar to Drive first'
    os.makedirs(f'{WORK}/data/IXI', exist_ok=True)
    !tar -xf {T1L_TAR} -C {WORK}/data/IXI
n = len(glob.glob(f'{T1L_DIR}/subjects/*/*/t1_linear/*_res-2x2x2_T1w.nii.gz'))
print(n, 't1-linear images (expected 563)')

## 4. Installer les dépendances

In [ ]:
!pip install -q clinicadl==2.0.0 xlrd
!python -c "import clinicadl, torch; print('ClinicaDL', clinicadl.__version__, '| CUDA', torch.cuda.is_available())"

## 5. Télécharger IXI (≈ 4,5 Go, quelques minutes sur Colab)

Source officielle : https://brain-development.org/ixi-dataset/. L'étape est sautée si les images sont déjà là.

In [ ]:
import glob
%cd {WORK}/data/IXI
URL = 'http://biomedic.doc.ic.ac.uk/brain-development/downloads/IXI'
if not os.path.exists('IXI.xls'):
    !curl -fL --retry 3 -o IXI.xls {URL}/IXI.xls
if len(glob.glob('raw/*.nii.gz')) < 500:
    !curl -fL --retry 3 -o IXI-T1.tar {URL}/IXI-T1.tar
    !tar -xf IXI-T1.tar -C raw && rm IXI-T1.tar
!ls raw | wc -l

## 6. Conversion en BIDS + fichier des âges

In [ ]:
%cd {WORK}
!python scripts/ixi_to_bids.py

## Pré-calcul du prétraitement (une fois par session)

Rééchantillonnage à 2 mm, normalisation et recadrage de toutes les images vers `data/IXI/bids_2mm`, pour ne pas les refaire à chaque epoch. Les images déjà traitées sont sautées.

In [ ]:
%cd {WORK}
!python scripts/cache_preprocessed.py

## 7. Contrôle du prétraitement (une image par hôpital)

In [ ]:
!python scripts/train_ixi_site_shift.py --check
from IPython.display import Image, display
for p in sorted(glob.glob(f'{WORK}/outputs/site_shift_test-IOP/check_*_sub-*.png')):
    display(Image(p))

## 7 bis. Archiver le run v1 et calculer ses prédictions signées

À faire **avant** de relancer un entraînement : le 1er run (MAE insite 6,16 / outofsite 12,21) est déplacé dans `site_shift_test-IOP_v1_lr1e-3` pour ne pas être effacé par `clean_run()`, puis on calcule ses prédictions signées. La cellule ne déplace rien si l'archive existe déjà.

In [ ]:
import shutil
RUN_DIR = f'{WORK}/outputs/site_shift_test-IOP'
ARCHIVE = f'{WORK}/outputs/site_shift_test-IOP_v1_lr1e-3'

# The archiving step is done (v2 now lives in RUN_DIR): this cell no longer moves anything.
if os.path.exists(ARCHIVE):
    !python scripts/predict_ages.py --run-dir {ARCHIVE}
    for p in sorted(glob.glob(f'{ARCHIVE}/*.png')):
        if 'check_' not in p:
            display(Image(p))
else:
    print('No v1 archive found: nothing to do.')

## 8. Entraînement : Guys + HH → test sur IOP

Depuis la v2 : l'âge est normalisé à l'intérieur du modèle et le learning rate vaut 1e-4 (modifiable avec `--lr`).

ClinicaDL refuse d'écraser un dossier MAPS ou un split existant : `clean_run()` les supprime avant chaque lancement (les images de contrôle sont gardées).

Commencer par `--epochs 2` pour vérifier, puis lancer le vrai entraînement.

In [ ]:
import shutil
RUN_DIR = f'{WORK}/outputs/site_shift_test-IOP'

def clean_run():
    for name in ['maps', 'split', 'train_sites.tsv', 'out_of_site.tsv']:
        path = os.path.join(RUN_DIR, name)
        if os.path.isdir(path):
            shutil.rmtree(path)
        elif os.path.exists(path):
            os.remove(path)

clean_run()
!python scripts/train_ixi_site_shift.py --test-site IOP --epochs 2 --batch-size 8

In [ ]:
clean_run()
!python scripts/train_ixi_site_shift.py --test-site IOP --epochs 50 --batch-size 8

## 9. Résultats

MAE (en années) sur le test interne (`in-site`, Guys + HH) et sur l'hôpital jamais vu (`out-of-site`, IOP).

In [ ]:
import pandas as pd
for group in ['insite', 'outofsite']:
    for tsv in sorted(glob.glob(f'{RUN_DIR}/maps/test/group-{group}/results/*/*/metrics/aggregated.tsv')):
        mae = pd.read_csv(tsv, sep='\t')['mae'].iloc[0]
        print(f'{group:12s} MAE = {mae:.2f} ans')

## 10. Prédictions signées (âge prédit − âge réel)

Recharge le modèle `best-mae` et calcule l'âge prédit de chaque sujet de test : biais par tranche d'âge, graphique prédit vs réel, courbe de validation. Tout est enregistré dans le dossier du run.

In [ ]:
!python scripts/predict_ages.py --test-site IOP
for p in sorted(glob.glob(f'{RUN_DIR}/*.png')):
    if 'check_' not in p:
        display(Image(p))

## 11. Swap the unseen hospital (HH, then Guys)

Same experiment with another hospital held out: train on the two others, test on `TEST_SITE`. Results go to `outputs/site_shift_test-<site>`; the IOP run (`site_shift_test-IOP`) is never touched.

Run this cell once with `TEST_SITE = 'HH'`, then once with `TEST_SITE = 'Guys'` (~20-25 min each).

In [ ]:
import shutil
TEST_SITE = 'HH'   # then 'Guys'
SITE_DIR = f'{WORK}/outputs/site_shift_test-{TEST_SITE}'
assert TEST_SITE in ['HH', 'Guys'], 'This cell must never touch the IOP run'

# only cleans the folder of THIS test site (a previous run on the same site is replaced)
for name in ['maps', 'split', 'train_sites.tsv', 'out_of_site.tsv']:
    path = os.path.join(SITE_DIR, name)
    if os.path.isdir(path):
        shutil.rmtree(path)
    elif os.path.exists(path):
        os.remove(path)

!python scripts/train_ixi_site_shift.py --test-site {TEST_SITE} --epochs 50 --batch-size 8
!python scripts/predict_ages.py --test-site {TEST_SITE}
for p in sorted(glob.glob(f'{SITE_DIR}/predictions_*.png')) + glob.glob(f'{SITE_DIR}/validation_curve.png'):
    display(Image(p))

## 12. Same experiments with scanner-contrast augmentation (v3)

During training only, each image gets a random gamma (contrast between tissues, 0.70-1.42) and a smooth bias field, so the model learns to ignore each scanner's contrast style. Validation and test images are never augmented.

Results go to `outputs/site_shift_test-<site>_aug`, next to the v2 runs, which are never touched. Run this cell three times: `TEST_SITE = 'IOP'`, then `'HH'`, then `'Guys'` (~20-25 min each).

In [ ]:
import shutil
TEST_SITE = 'IOP'   # then 'HH', then 'Guys'
AUG_DIR = f'{WORK}/outputs/site_shift_test-{TEST_SITE}_aug'
assert AUG_DIR.endswith('_aug'), 'This cell must only touch *_aug folders'

# only cleans the *_aug folder of THIS test site (the v2 runs without augmentation are never touched)
for name in ['maps', 'split', 'train_sites.tsv', 'out_of_site.tsv']:
    path = os.path.join(AUG_DIR, name)
    if os.path.isdir(path):
        shutil.rmtree(path)
    elif os.path.exists(path):
        os.remove(path)

!python scripts/train_ixi_site_shift.py --test-site {TEST_SITE} --augment --epochs 50 --batch-size 8
!python scripts/predict_ages.py --test-site {TEST_SITE} --augment
for p in sorted(glob.glob(f'{AUG_DIR}/predictions_*.png')) + glob.glob(f'{AUG_DIR}/validation_curve.png'):
    display(Image(p))

## 13. Same experiments on Clinica t1-linear images

Images preprocessed with Clinica t1-linear (N4 bias correction + affine registration to MNI, then crop), unpacked by cell 3 bis. At load time they are padded to 96x112x96 and z-normalized.

For a session with **only** these runs, cells 2, 3, 3 bis, 4 and then this section are enough: no need to download IXI (cells 5-6) or to rebuild the cache.

Results go to `outputs/site_shift_test-<site>_t1linear` (and `..._t1linear_aug` with `AUGMENT = True`): the earlier runs are never touched. Run the training cell for `TEST_SITE = 'IOP'`, then `'HH'`, then `'Guys'`.

In [ ]:
import glob, os
from IPython.display import Image, display
%cd {WORK}
# one image per hospital: check the framing before training
!python scripts/train_ixi_site_shift.py --check --data t1linear
for p in sorted(glob.glob(f'{WORK}/outputs/site_shift_test-IOP_t1linear/check_*.png')):
    display(Image(p))

In [ ]:
import shutil
TEST_SITE = 'IOP'   # then 'HH', then 'Guys'
AUGMENT = False     # True: same run with scanner-contrast augmentation
T1_DIR = f"{WORK}/outputs/site_shift_test-{TEST_SITE}_t1linear{'_aug' if AUGMENT else ''}"
assert '_t1linear' in T1_DIR, 'This cell must only touch *_t1linear folders'
AUG_FLAG = '--augment' if AUGMENT else ''

# only cleans the folder of THIS run (the bids2mm runs are never touched)
for name in ['maps', 'split', 'train_sites.tsv', 'out_of_site.tsv']:
    path = os.path.join(T1_DIR, name)
    if os.path.isdir(path):
        shutil.rmtree(path)
    elif os.path.exists(path):
        os.remove(path)

!python scripts/train_ixi_site_shift.py --data t1linear --test-site {TEST_SITE} {AUG_FLAG} --epochs 50 --batch-size 8
!python scripts/predict_ages.py --data t1linear --test-site {TEST_SITE} {AUG_FLAG}
for p in sorted(glob.glob(f'{T1_DIR}/predictions_*.png')) + glob.glob(f'{T1_DIR}/validation_curve.png'):
    display(Image(p))

### Summary of all runs

In-site / out-of-site MAE (years) of every finished run, read from the MAPS test results.

In [ ]:
import pandas as pd
rows = []
for run in sorted(glob.glob(f'{WORK}/outputs/site_shift_test-*')):
    row = {'run': os.path.basename(run).removeprefix('site_shift_test-')}
    for group in ['insite', 'outofsite']:
        tsvs = glob.glob(f'{run}/maps/test/group-{group}/results/*/*/metrics/aggregated.tsv')
        if tsvs:
            row[group] = round(pd.read_csv(tsvs[0], sep='	')['mae'].iloc[0], 2)
    if len(row) > 1:
        rows.append(row)
pd.DataFrame(rows)